# 2회차 · 지도학습 1 — 정답이 있는 데이터로 규칙 배우기
**2026 경기도 공유학교 · 인공지능 기초 (김순찬 교수)**

## 오늘의 학습 목표
1. 지도학습이 "정답(라벨)이 붙은 데이터로 규칙을 배우는 것"임을 설명할 수 있다.
2. **회귀**(숫자 맞히기)와 **분류**(종류 맞히기)를 구분할 수 있다.
3. 선형회귀 · kNN · 결정트리를 직접 학습시키고, 결과를 읽을 수 있다.
4. 문제 상황을 보고 알맞은 알고리즘을 고르고 **근거**를 말할 수 있다.

## 사용법
- 셀 왼쪽의 ▶ 버튼을 누르거나 **Shift + Enter** 를 누르면 그 셀이 실행됩니다.
- 반드시 **위에서부터 순서대로** 실행하세요. (앞 셀에서 만든 변수를 뒤 셀이 씁니다.)
- 오류가 나면 당황하지 말고: ① 빨간 글씨 **마지막 줄**을 읽는다 → ② 앞 셀을 실행했는지 확인 → ③ 안 되면 **런타임 → 런타임 다시 시작** 후 처음부터.
- 🧩 **직접 해보기** 셀은 `# TODO` 부분을 채워서 실행합니다. 막히면 바로 아래 "정답 보기"를 펼치세요.

> 이 노트북은 인터넷에서 파일을 내려받지 않습니다. 데이터는 코드 안에서 만들거나(합성), scikit-learn에 들어 있는 붓꽃(iris) 데이터를 씁니다.

## 0. 준비 — 도구 불러오기
파이썬에서 "도구 상자"를 여는 것을 `import` 라고 합니다.
- `numpy` : 숫자 계산 (배열)
- `pandas` : 표(데이터프레임) 다루기
- `matplotlib` : 그래프 그리기
- 아래 셀은 Colab에 **한글 폰트**도 설치합니다(그래프에 한글이 □□로 깨지지 않게). 10초쯤 걸릴 수 있어요.

**확인할 것:** 마지막 줄에 `준비 완료!` 가 나오면 성공.

In [ ]:
import matplotlib.pyplot as plt, matplotlib as mpl, numpy as np, pandas as pd
import subprocess, sys
try:
    subprocess.run(['apt-get','-qq','install','-y','fonts-nanum'], check=False, capture_output=True)   # Colab 한글 폰트
    import matplotlib.font_manager as fm; fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf'); plt.rcParams['font.family']='NanumGothic'
except Exception:
    plt.rcParams['font.family'] = 'Noto Sans CJK JP'   # 로컬 검증용
plt.rcParams['axes.unicode_minus'] = False
print("준비 완료! numpy", np.__version__, "/ pandas", pd.__version__)

---
# Part A. 회귀 — 공부 시간으로 성적을 맞혀 보자
**회귀(regression)** 는 맞힐 것이 **숫자**일 때 쓰는 지도학습입니다. (예: 내일 기온, 집값, 시험 점수)

## A-1. 데이터 만들기
40명의 학생에 대해 `공부시간`(하루 평균, 시간)과 `성적`(점)을 기록한 가상의 표를 만듭니다.
- `np.random.default_rng(2024)` : 난수 생성기에 **씨앗(seed)** 2024를 줍니다. 씨앗이 같으면 누가 실행해도 **똑같은** 데이터가 나옵니다.
- 성적은 "42점 + 공부시간×7.5 + 약간의 운(잡음)" 으로 만들었습니다. 우리는 이 규칙을 **모른다고 치고**, 모델이 데이터에서 찾아내는지 볼 겁니다.
- `df.head()` 는 표의 **앞 5줄**을 보여 줍니다.

**확인할 것:** 열이 2개(공부시간, 성적), 행이 40개.

In [ ]:
rng = np.random.default_rng(2024)                       # 씨앗 고정 → 항상 같은 데이터
hours = np.round(rng.uniform(0.5, 6, 40), 1)             # 하루 공부시간 0.5~6시간, 40명
score = np.clip(np.round(42 + 7.5 * hours + rng.normal(0, 6, 40)), 0, 100)   # 성적 (0~100점)
df = pd.DataFrame({'공부시간': hours, '성적': score})   # 표로 만들기
print("행, 열 =", df.shape)
df.head()

## A-2. 그림으로 먼저 보기 — 산점도
모델을 만들기 전에 **항상 그림을 먼저** 봅니다. 점이 대략 직선처럼 늘어서 있으면 "선형회귀"가 잘 맞을 후보입니다.
- `plt.scatter(x, y)` : x, y 좌표에 점 찍기
- `plt.xlabel / ylabel` : 축 이름 (단위까지 쓰는 습관!)

**확인할 것:** 공부시간이 늘수록 성적이 대체로 올라가는가? 완벽한 직선인가, 흩어져 있는가?

In [ ]:
plt.figure(figsize=(6, 4))
plt.scatter(df['공부시간'], df['성적'], color='#4F5BD5')
plt.xlabel('하루 공부 시간 (시간)'); plt.ylabel('시험 점수 (점)')
plt.title('공부 시간과 성적 (학생 40명)')
plt.grid(alpha=.3); plt.show()

## A-3. 선형회귀 모델 학습시키기
**선형회귀**는 점들 사이를 지나는 **가장 오차가 작은 직선** 하나를 찾습니다. 직선은 `성적 = 기울기 × 공부시간 + 절편` 꼴입니다.

scikit-learn(사이킷런)에서 모델을 쓰는 순서는 **항상 같습니다**:
1. `model = LinearRegression()` — 빈 모델 만들기
2. `model.fit(X, y)` — 데이터로 **학습**(fit = 맞추다). `X`는 입력(특징), `y`는 정답.
3. `model.predict(새 X)` — **예측**

주의: 입력 `X`는 **2차원 표** 모양이어야 합니다. 그래서 `df[['공부시간']]` 처럼 **대괄호를 두 번** 씁니다(열이 하나라도 표). 정답 `y`는 `df['성적']` 처럼 한 번만.

**확인할 것:** 기울기가 약 7.5, 절편이 약 42 근처로 나오나요? (데이터를 만들 때 넣은 규칙과 비교)

In [ ]:
from sklearn.linear_model import LinearRegression

X = df[['공부시간']]     # 입력(특징): 대괄호 2개 → 표 모양 (40행 1열)
y = df['성적']          # 정답(라벨)

model = LinearRegression()   # 1) 빈 모델
model.fit(X, y)              # 2) 학습 — 오차가 가장 작은 직선 찾기

print("기울기 (공부 1시간당 오르는 점수):", round(model.coef_[0], 2))
print("절편   (공부 0시간일 때 점수)    :", round(model.intercept_, 2))

## A-4. 찾아낸 직선을 점 위에 그려 보기
모델이 찾은 직선이 점들 사이를 잘 지나가는지 눈으로 확인합니다.
- `np.linspace(0, 7, 50)` : 0부터 7까지 50개의 x값을 고르게 만들기
- 그 x값들을 모델에 넣어 예측값(직선 위의 y)을 구하고 `plt.plot` 으로 선을 긋습니다.
- `pd.DataFrame({'공부시간': xs})` 로 만드는 이유: 학습할 때와 **같은 열 이름의 표** 모양으로 넣어야 경고가 안 뜹니다.

**확인할 것:** 직선 위·아래로 점이 고르게 흩어져 있나요? (한쪽으로만 몰리면 직선이 안 맞는 것)

In [ ]:
xs = np.linspace(0, 7, 50)                                  # 0~7시간 사이 50개 점
ys = model.predict(pd.DataFrame({'공부시간': xs}))           # 직선 위의 예측값

plt.figure(figsize=(6, 4))
plt.scatter(df['공부시간'], df['성적'], color='#4F5BD5', label='실제 학생')
plt.plot(xs, ys, color='#C0395F', linewidth=2.5, label='모델이 찾은 직선')
plt.xlabel('하루 공부 시간 (시간)'); plt.ylabel('시험 점수 (점)')
plt.title('선형회귀 결과'); plt.legend(); plt.grid(alpha=.3); plt.show()

## A-5. 예측해 보기 — "하루 3시간 공부하면 몇 점?"
학습이 끝난 모델에 **새로운 입력**을 넣으면 예측값을 돌려줍니다. 이것이 지도학습의 목적입니다.
- 여러 명을 한 번에 넣을 수 있습니다(표의 행 하나 = 학생 한 명).
- 예측은 **"이 데이터 범위 안에서"** 믿을 만합니다. 20시간을 넣으면 100점을 넘는 엉터리 값이 나올 수 있어요(하루는 24시간!). 이런 걸 **외삽(extrapolation)의 위험**이라고 합니다.

**확인할 것:** 공부시간이 1시간 늘 때 예측 점수가 기울기만큼(약 7.5점) 늘어나는가?

In [ ]:
new = pd.DataFrame({'공부시간': [1, 2, 3, 4, 5]})
pred = model.predict(new)
for h, p in zip(new['공부시간'], pred):
    print(f"하루 {h}시간 공부 → 예측 점수 {p:.1f}점")

print("\n(참고) 20시간을 넣으면:", round(model.predict(pd.DataFrame({'공부시간': [20]}))[0], 1), "점 ← 100점을 넘는 엉터리 값! 데이터 범위 밖 예측은 위험")

## 🧩 직접 해보기 1 — 내 공부 시간으로 예측
`# TODO` 자리에 **여러분의 하루 평균 공부 시간**(예: 2.5)을 넣고 실행해 보세요. 그리고 실제 내 성적과 비교해 보세요. 왜 다를까요? (모델은 "공부시간" 하나만 보고 있다는 점을 생각해 보세요.)

In [ ]:
my_hours = 2.5   # TODO: 여러분의 하루 평균 공부 시간으로 바꾸세요 (예: 1.5, 3, 4.5)
my_pred = model.predict(pd.DataFrame({'공부시간': [my_hours]}))[0]
print(f"하루 {my_hours}시간 공부 → 모델의 예측 점수: {my_pred:.1f}점")

<details><summary>정답 보기</summary>

`my_hours = 3.0` 처럼 숫자만 바꾸면 됩니다. 예측이 실제와 다른 이유: 모델은 **공부시간 하나만** 알고, 수면·집중력·과목 난이도 같은 다른 정보는 모릅니다. 또 이 데이터는 가상의 40명 자료라서 여러분 학교와 다를 수 있어요. → **입력 특징을 더 넣거나(3회차), 더 많은 데이터**를 모아야 정확해집니다.
</details>

---
# Part B. 분류 — 꽃잎 크기로 붓꽃 종류 맞히기
**분류(classification)** 는 맞힐 것이 **종류(카테고리)** 일 때 쓰는 지도학습입니다. (예: 스팸/정상 메일, 고양이/강아지 사진, 합격/불합격)

## B-1. 붓꽃(iris) 데이터 불러오기
scikit-learn에 들어 있는 유명한 데이터입니다. 붓꽃 **150송이**의 꽃받침·꽃잎 길이/너비(4가지 특징)와 **품종(3종)** 이 기록되어 있습니다.
- `load_iris()` : 데이터 불러오기 (인터넷 필요 없음)
- `iris.data` : 특징 4개(숫자), `iris.target` : 정답(0=setosa, 1=versicolor, 2=virginica)
- 보기 편하게 `DataFrame` 표로 바꾸고 열 이름을 한글로 붙입니다.

**확인할 것:** 150행 × 5열(특징 4 + 품종 1), 품종별로 50송이씩.

In [ ]:
from sklearn.datasets import load_iris
iris = load_iris()

iris_df = pd.DataFrame(iris.data, columns=['꽃받침길이', '꽃받침너비', '꽃잎길이', '꽃잎너비'])
iris_df['품종'] = [iris.target_names[t] for t in iris.target]   # 0,1,2 → 이름

print("행, 열 =", iris_df.shape)
print(iris_df['품종'].value_counts())
iris_df.sample(5, random_state=1)     # 무작위로 5줄 보기

## B-2. 그림으로 먼저 보기 — 꽃잎 길이 vs 꽃잎 너비
품종별로 색을 다르게 해서 산점도를 그립니다. **분류가 쉬운지 어려운지**는 그림에서 대략 보입니다.
- 세 덩어리가 확실히 떨어져 있으면 쉬운 문제, 겹치면 어려운 문제.

**확인할 것:** setosa는 완전히 따로 떨어져 있나요? versicolor와 virginica는 조금 겹치나요?

In [ ]:
colors = {'setosa': '#0E8C86', 'versicolor': '#D97706', 'virginica': '#C0395F'}
plt.figure(figsize=(6, 4.2))
for name, g in iris_df.groupby('품종'):
    plt.scatter(g['꽃잎길이'], g['꽃잎너비'], color=colors[name], label=name, s=30)
plt.xlabel('꽃잎 길이 (cm)'); plt.ylabel('꽃잎 너비 (cm)')
plt.title('붓꽃 3종 — 꽃잎 크기'); plt.legend(); plt.grid(alpha=.3); plt.show()

## B-3. 훈련용 / 테스트용으로 나누기 (매우 중요!)
공부한 문제집으로 시험을 보면 점수가 당연히 높겠죠? 모델도 마찬가지입니다. 그래서 데이터를 **둘로 나눕니다**:
- **훈련(train) 데이터** : 모델이 공부하는 문제집 (75%)
- **테스트(test) 데이터** : 처음 보는 시험지 (25%) — 학습에는 **절대** 쓰지 않음

`train_test_split(X, y, test_size=0.25, random_state=0)`
- `test_size=0.25` : 25%를 테스트용으로
- `random_state=0` : 섞는 방식을 고정 → 누가 해도 같은 결과
- `stratify=y` : 품종 비율을 훈련/테스트에서 똑같이 유지

**확인할 것:** 훈련 112송이, 테스트 38송이로 나뉘었나요?

In [ ]:
from sklearn.model_selection import train_test_split

X = iris_df[['꽃받침길이', '꽃받침너비', '꽃잎길이', '꽃잎너비']]   # 특징 4개
y = iris_df['품종']                                                   # 정답

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
print("훈련용:", len(X_train), "송이 / 테스트용:", len(X_test), "송이")

## B-4. kNN — 가까운 이웃에게 물어보기
**k-최근접 이웃(kNN)** 은 새 꽃이 오면 특징이 가장 비슷한(거리가 가까운) **k송이**를 찾아, 그중 **가장 많은 품종**으로 답합니다. 규칙을 따로 외우지 않고 "이웃 투표"로 정하는 단순한 방법입니다.
- `KNeighborsClassifier(n_neighbors=3)` : k=3 (이웃 3송이)
- `fit` 으로 훈련 데이터를 **기억**시키고, `score(X_test, y_test)` 로 테스트 정확도(맞힌 비율)를 구합니다.
- k를 3과 5로 바꿔 비교해 봅니다.

**확인할 것:** 정확도가 0.9(90%) 이상 나오나요? k=3과 k=5는 같은가요, 다른가요?

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

for k in [3, 5]:
    knn = KNeighborsClassifier(n_neighbors=k)   # 이웃 k송이
    knn.fit(X_train, y_train)                   # 훈련 데이터 기억
    acc = knn.score(X_test, y_test)             # 테스트 정확도 = 맞힌 개수 / 전체
    print(f"kNN (k={k}) 테스트 정확도: {acc:.3f}  → 38송이 중 {round(acc*38)}송이 맞힘")

## B-5. 결정트리 — "예/아니오" 질문으로 나누기
**결정트리**는 스무고개처럼 "꽃잎 길이가 2.5cm 이하인가?" 같은 질문을 이어 가며 데이터를 나눕니다. 장점은 **왜 그렇게 판단했는지 눈으로 볼 수 있다**는 것.
- `max_depth=3` : 질문을 최대 3단계까지만 (너무 깊으면 훈련 데이터를 통째로 외워 버림 — 3회차에서 자세히)
- `plot_tree` 로 배운 규칙을 그림으로 봅니다. 각 상자: 질문 / 그 상자에 온 송이 수 / 품종별 개수 / 다수 품종.

**확인할 것:** 맨 위 질문이 "꽃잎"에 관한 것인가요? 왼쪽 가지로 가면 무슨 품종이 되나요?

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

tree = DecisionTreeClassifier(max_depth=3, random_state=0)   # 질문 최대 3단계
tree.fit(X_train, y_train)
print("결정트리 테스트 정확도:", round(tree.score(X_test, y_test), 3))

plt.figure(figsize=(13, 6))
plot_tree(tree, feature_names=list(X.columns), class_names=list(tree.classes_), filled=True, rounded=True, fontsize=9)
plt.title('결정트리가 배운 규칙 (위에서 아래로 질문을 따라가세요)'); plt.show()

## B-6. 모델끼리 비교표 만들기
같은 테스트 데이터로 세 모델의 정확도를 한 표에 모읍니다. 이렇게 **같은 시험지로 비교**해야 공정합니다.

**확인할 것:** 어느 모델이 가장 높은가요? 차이가 크게 나나요? (iris는 쉬운 데이터라 대부분 90% 이상입니다. 3회차에서 정확도 말고 다른 평가 지표도 배웁니다.)

In [ ]:
models = {
    'kNN (k=3)': KNeighborsClassifier(n_neighbors=3),
    'kNN (k=5)': KNeighborsClassifier(n_neighbors=5),
    '결정트리 (깊이 3)': DecisionTreeClassifier(max_depth=3, random_state=0),
}
rows = []
for name, m in models.items():
    m.fit(X_train, y_train)
    rows.append({'모델': name, '훈련 정확도': round(m.score(X_train, y_train), 3), '테스트 정확도': round(m.score(X_test, y_test), 3)})
result = pd.DataFrame(rows)
result

## B-7. 새 꽃 예측하기
들판에서 꽃 한 송이를 재 왔다고 합시다: 꽃받침 5.0×3.4cm, 꽃잎 1.5×0.2cm. 어떤 품종일까요?
- 학습할 때와 **같은 열 순서**로 표를 만들어 넣습니다.
- `predict_proba` 는 kNN이 "이웃 투표 결과"를 확률로 보여 줍니다 (예: 5표 중 5표 → 1.0).

**확인할 것:** 세 모델의 답이 같은가요? 위 산점도에서 (1.5, 0.2) 위치가 어느 색 근처인지 찾아보세요.

In [ ]:
new_flower = pd.DataFrame({'꽃받침길이': [5.0, 6.7], '꽃받침너비': [3.4, 3.0], '꽃잎길이': [1.5, 5.2], '꽃잎너비': [0.2, 2.3]})
print(new_flower, "\n")
for name, m in models.items():
    print(f"{name:14s} → 예측:", [str(v) for v in m.predict(new_flower)])

knn5 = models['kNN (k=5)']
print("\nkNN(k=5) 투표 확률 (열 순서:", [str(c) for c in knn5.classes_], ")")
print(np.round(knn5.predict_proba(new_flower), 2))

## 🧩 직접 해보기 2 — k를 바꿔 가며 정확도 관찰
`# TODO` 부분의 리스트에 k값을 더 넣어(예: 1, 3, 5, 9, 15, 31) 테스트 정확도가 어떻게 변하는지 그래프로 그려 보세요.
- k=1이면 "가장 가까운 딱 한 송이"만 보고 답합니다 → 튀는 값(이상치)에 흔들리기 쉽습니다.
- k가 너무 크면(예: 100) 거의 모든 꽃을 보게 되어 "가장 흔한 품종"만 답하게 됩니다.

In [ ]:
ks = [1, 3, 5]   # TODO: k값을 더 넣어 보세요. 예) [1, 3, 5, 9, 15, 31]
accs = []
for k in ks:
    m = KNeighborsClassifier(n_neighbors=k).fit(X_train, y_train)
    accs.append(m.score(X_test, y_test))

plt.figure(figsize=(6, 3.5))
plt.plot(ks, accs, marker='o', color='#4F5BD5')
plt.xlabel('k (이웃 수)'); plt.ylabel('테스트 정확도'); plt.title('k에 따른 kNN 정확도'); plt.grid(alpha=.3); plt.show()
for k, a in zip(ks, accs): print(f"k={k:3d} → {a:.3f}")

<details><summary>정답 보기</summary>

`ks = [1, 3, 5, 9, 15, 31]` 로 바꾸면 됩니다. iris는 쉬운 데이터라 k가 어지간하면 정확도가 비슷하게 높습니다. 그래도 k가 매우 커지면(예: 100 이상) 정확도가 떨어지기 시작합니다. **"k를 어떻게 정할까?"** 는 3회차(모델 평가·튜닝)의 핵심 주제입니다.
</details>

---
# 📝 오늘의 결과물 제출 — 문제 상황에 알맞은 알고리즘 고르기
아래 세 가지 문제 상황을 읽고, 각각 **회귀/분류** 중 무엇인지, 어떤 알고리즘을 쓰겠는지, 그 **근거**를 씁니다. 답을 딕셔너리(`{ }`) 안의 따옴표 사이에 적고 실행하면 제출용 요약이 출력됩니다. (출력을 캡처해서 제출)

| 번호 | 문제 상황 |
|---|---|
| 1 | 우리 학교 매점에서 **내일 팔릴 삼각김밥 개수**를 요일·날씨·기온으로 맞히고 싶다. |
| 2 | 사진 속 급식 반찬이 **김치/단무지/깍두기 중 무엇인지** 색·모양 특징으로 맞히고 싶다. 왜 그렇게 판단했는지 규칙을 설명해야 한다. |
| 3 | 학생의 출석률·과제 제출률·중간 점수로 **기말 시험 통과 여부(예/아니오)** 와 그 **확률**을 알고 싶다. |

힌트: 숫자를 맞히면 **회귀**, 종류를 맞히면 **분류**. 규칙을 설명해야 하면 **결정트리**, 확률이 필요하면 **로지스틱 회귀**, 관계가 직선에 가까우면 **선형회귀**, 비슷한 것끼리 묶어 판단하려면 **kNN**.

In [ ]:
이름 = "홍길동"      # TODO: 이름
학번 = "10101"       # TODO: 학번

답안 = {
    1: {"유형": "회귀",   "알고리즘": "선형회귀",     "근거": "팔릴 개수는 숫자이고, 기온·요일과 대체로 직선 관계일 것 같아서"},   # TODO: 수정
    2: {"유형": "분류",   "알고리즘": "결정트리",     "근거": "종류를 맞히는 문제이고, 어떤 특징으로 판단했는지 규칙을 보여 줘야 해서"},   # TODO: 수정
    3: {"유형": "분류",   "알고리즘": "로지스틱 회귀", "근거": "예/아니오 두 가지를 맞히고 확률도 필요해서"},   # TODO: 수정
}

print("=" * 60)
print(f"2회차 지도학습 1 — 결과물 | {이름} ({학번})")
print("=" * 60)
print(f"[Part A] 선형회귀 기울기 {model.coef_[0]:.2f}, 절편 {model.intercept_:.2f}")
print(f"[Part B] 테스트 정확도 — " + ", ".join(f"{r['모델']} {r['테스트 정확도']}" for r in rows))
print("[결과물] 문제 상황별 알고리즘 선택")
for i, a in 답안.items():
    print(f"  {i}. {a['유형']} / {a['알고리즘']}  ← {a['근거']}")
print("=" * 60)

## ✅ 자기 점검 체크리스트
- [ ] 지도학습에서 "정답(라벨)"이 무엇인지 예를 들어 말할 수 있다.
- [ ] 회귀와 분류의 차이를 한 문장으로 말할 수 있다.
- [ ] `fit` 은 학습, `predict` 는 예측이라는 것을 안다.
- [ ] 왜 훈련/테스트 데이터를 나누는지 설명할 수 있다.
- [ ] kNN의 k, 결정트리의 max_depth 가 무엇을 조절하는지 안다.
- [ ] 문제 상황 3개에 알맞은 알고리즘을 고르고 근거를 썼다.

**다음 시간(3회차):** 정확도만 믿으면 안 되는 이유 · 과적합 · 여러 지표로 모델을 평가하고 **더 좋게 튜닝**하기